Ячейка 1: Инициализация проекта и импорт библиотек

In [1]:
import os
import json
import re
import pandas as pd
from pathlib import Path
from datetime import datetime

# Определяем пути к данным
RAW_DATA_DIR = Path("../data/raw_things/")
OUTPUT_DIR = Path("../data/output/")

# Создаем папку для выгрузки, если её еще нет
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Форматируем текущую дату и время (ГодМесяцДень_ЧасыМинутыСекунды)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
output_file = OUTPUT_DIR / f"shields_result_{timestamp}.csv"

print(f"Библиотеки импортированы. Рабочие директории настроены.")
print(f"Файл при экспорте будет сохранен как: {output_file.name}")

Библиотеки импортированы. Рабочие директории настроены.
Файл при экспорте будет сохранен как: shields_result_20260726_185131.csv


Ячейка 2: Загрузка баз данных и построение глобального словаря перевода

In [2]:
# Загружаем базы данных
with open(RAW_DATA_DIR / "NexusConfigStoreInventory.json", "r", encoding="utf-8") as f:
    inventory_data = json.load(f)
with open(RAW_DATA_DIR / "NexusConfigStoreInventoryNamePart.json", "r", encoding="utf-8") as f:
    name_parts_data = json.load(f)
with open(RAW_DATA_DIR / "NexusConfigStore_StatDisplay.json", "r", encoding="utf-8") as f:
    stat_display_data = json.load(f)

# Загружаем также базу данных прошивок
firmware_path = RAW_DATA_DIR / "NexusConfigStoreFirmware.json"
firmware_data = {}
if firmware_path.exists():
    with open(firmware_path, "r", encoding="utf-8") as f:
        firmware_data = json.load(f)

def clean_bbcode(text):
    if not text: return ""
    cleaned = re.sub(r"\[.*?\]", "", text)
    if " - " in cleaned:
        cleaned = cleaned.split(" - ")[0]
    return cleaned.strip()

# Строим глобальную карту перевода для абсолютно всех деталей игры
part_translation_map = {}

# 1. Автоматический перевод через NamePart и StatDisplay (по всем категориям инвентаря)
for category, cat_val in inventory_data.items():
    if not isinstance(cat_val, dict): continue
    parts_dict = cat_val.get("parts", {})
    if not isinstance(parts_dict, dict): continue
        
    for part_id, part_val in parts_dict.items():
        if not isinstance(part_val, dict): continue
        part_name = part_val.get("name", "")
        if not isinstance(part_name, str) or not part_name.startswith("part_"):
            continue
        
        fields = part_val.get("fields", {})
        if not isinstance(fields, dict): continue
            
        aspects = fields.get("Aspects", [])
        for aspect in aspects:
            if not isinstance(aspect, dict): continue
            
            # Вариант А: Ищем в аспектах именования InventoryNamingAspect
            if "InventoryNamingAspect" in aspect.get("structtype", ""):
                title_list = aspect.get("TitlePartList", []) or []
                prefix_list = aspect.get("PrefixPartList", []) or []
                suffix_list = aspect.get("SuffixPartList", []) or []
                for naming_def in title_list + prefix_list + suffix_list:
                    if isinstance(naming_def, str):
                        match = re.search(r"'(np_.*?)'", naming_def)
                        if match:
                            np_key = match.group(1)
                            if np_key in name_parts_data:
                                real_name = name_parts_data[np_key].get("fields", {}).get("PartName", "")
                                if real_name:
                                    part_translation_map[part_name.lower()] = real_name
                                    part_translation_map[part_name.replace("part_", "").lower()] = real_name
                                    
            # Вариант Б: Ищем в аспектах интерфейса UIStatAspect (без жесткого ограничения _lp_)
            elif "UIStatAspect" in aspect.get("structtype", ""):
                ui_stats = aspect.get("UIStatsToInclude", []) or []
                for stat_def in ui_stats:
                    if isinstance(stat_def, str):
                        match = re.search(r"'(uistat_.*?)'", stat_def)
                        if match:
                            lp_key = match.group(1)
                            if lp_key in stat_display_data:
                                format_text = stat_display_data[lp_key].get("fields", {}).get("StatValue", {}).get("FormatText", "")
                                if format_text:
                                    real_lp_name = clean_bbcode(format_text)
                                    if real_lp_name:
                                        # Очищаем системное форматирование движка (+{Value}%)
                                        real_lp_name = re.sub(r"\+?\{.*?\}%?\s*", "", real_lp_name).strip()
                                        if real_lp_name:
                                            part_translation_map[part_name.lower()] = real_lp_name
                                            part_translation_map[part_name.replace("part_", "").lower()] = real_lp_name

# 2. Добавляем перевод прошивок напрямую из FirmwareDef
for fw_key, fw_val in firmware_data.items():
    if isinstance(fw_val, dict):
        fw_name = fw_val.get("fields", {}).get("Name", "")
        if fw_name:
            fw_key_clean = fw_key.lower().replace("fw_", "")
            part_translation_map[fw_key.lower()] = fw_name
            part_translation_map[f"part_firmware_{fw_key_clean}"] = fw_name
            part_translation_map[f"part_firmware_{fw_key.lower()}"] = fw_name

print(f"Базы данных загружены! Построен автоматический словарь перевода деталей: {len(part_translation_map)} записей.")

Базы данных загружены! Построен автоматический словарь перевода деталей: 2044 записей.


Ячейка 3: Сбор легендарных и перламутровых щитов

In [3]:
legendary_shields = []

# Фильтруем только оружейные папки щитов
shield_categories = [cat for cat in inventory_data.keys() if "shield" in cat.lower()]

for category in shield_categories:
    cat_val = inventory_data[category]
    parts_dict = cat_val.get("parts", {})
    if not isinstance(parts_dict, dict):
        continue
        
    for part_id, part_val in parts_dict.items():
        if not isinstance(part_val, dict):
            continue
            
        part_path = part_val.get("path", "")
        if not isinstance(part_path, str):
            continue
            
        is_legendary = "comp_05_legendary" in part_path
        is_pearlescent = "comp_06_pearl" in part_path
        
        if is_legendary or is_pearlescent:
            fields = part_val.get("fields", {})
            if not isinstance(fields, dict):
                continue
                
            is_exclude = fields.get("bExcludeFromGlobalPool", False)
            world_drop_flag = not is_exclude
            
            selection_rules = fields.get("PartTypeSelectionRules", {})
            if not isinstance(selection_rules, dict):
                selection_rules = {}
                
            rarity_str = "Pearlescent" if is_pearlescent else "Legendary"
            
            legendary_shields.append({
                "Item_Code": part_path,
                "Internal_Category": category,
                "Rarity": rarity_str,
                "World_Drop": world_drop_flag,
                "Manufacturer": "Unknown",
                "Display_Name": "Unknown",
                "Drop_Source": "Unknown",
                "Selection_Rules": selection_rules
            })

df = pd.DataFrame(legendary_shields)
df = df.drop_duplicates(subset=["Item_Code"]).reset_index(drop=True)

print(f"Инициализация щитов завершена. Собрано уникальных щитов: {len(df)}")
df.head()

Инициализация щитов завершена. Собрано уникальных щитов: 32


,Item_Code,Internal_Category,Rarity,World_Drop,Manufacturer,Display_Name,Drop_Source,Selection_Rules
0,Armor_Shield.comp_05_legendary,237 | Armor_Shield,Legendary,True,Unknown,Unknown,Unknown,{}
1,Shield.comp_05_legendary,246 | Shield,Legendary,False,Unknown,Unknown,Unknown,"{'firmware': {'PartCount': {'min': 0, 'MAX': 1..."
2,energy_shield.comp_05_legendary,248 | energy_shield,Legendary,True,Unknown,Unknown,Unknown,{}
3,mal_shield.comp_05_legendary,279 | mal_shield,Legendary,False,Unknown,Unknown,Unknown,{}
4,vla_shield.comp_05_legendary,283 | vla_shield,Legendary,False,Unknown,Unknown,Unknown,{}


Ячейка 4: Определение производителей

In [4]:
# Словарь соответствия префиксов производителей щитов
mfr_shield_mapping = {
    "DAD": "Daedalus", "ORD": "Order", "BORG": "Ripper", "BOR": "Ripper",
    "JAK": "Jakobs", "VLA": "Vladof", "MAL": "Maliwan", "HYP": "Hyperion",
    "TED": "Tediore", "TOR": "Torgue", "COV": "CoV", "ATL": "Atlas"
}

def determine_shield_manufacturer(item_code):
    if not isinstance(item_code, str):
        return "Unknown"
    prefix = item_code.split("_")[0].upper()
    prefix = prefix.replace("INV'", "").replace("'", "")
    return mfr_shield_mapping.get(prefix, "Unknown")

df["Manufacturer"] = df["Item_Code"].apply(determine_shield_manufacturer)
print("Производители щитов успешно привязаны!")
df[["Item_Code", "Manufacturer"]].head()

Производители щитов успешно привязаны!


,Item_Code,Manufacturer
0,Armor_Shield.comp_05_legendary,Unknown
1,Shield.comp_05_legendary,Unknown
2,energy_shield.comp_05_legendary,Unknown
3,mal_shield.comp_05_legendary,Maliwan
4,vla_shield.comp_05_legendary,Vladof


Ячейка 5: Расшифровка оригинальных имен щитов

In [5]:
def resolve_display_name(item_code, name_parts):
    if not isinstance(item_code, str):
        return "Unknown"
        
    match = re.search(r"comp_05_legendary_(.*)", item_code, re.IGNORECASE)
    if not match:
        match = re.search(r"comp_06_pearl_(.*)", item_code, re.IGNORECASE)
        
    if match:
        raw_name = match.group(1).lower().replace("_", "")
        for np_key, np_val in name_parts.items():
            clean_np_key = np_key.lower().replace("np_", "").replace("_", "")
            if clean_np_key == raw_name or clean_np_key.endswith(raw_name):
                return np_val.get("fields", {}).get("PartName", "Unknown")
                
    return "Unknown"

df["Display_Name"] = df.apply(lambda row: resolve_display_name(row["Item_Code"], name_parts_data), axis=1)
print("Названия щитов успешно расшифрованы!")
df[["Item_Code", "Display_Name"]].head()

Названия щитов успешно расшифрованы!


,Item_Code,Display_Name
0,Armor_Shield.comp_05_legendary,Unknown
1,Shield.comp_05_legendary,Unknown
2,energy_shield.comp_05_legendary,Unknown
3,mal_shield.comp_05_legendary,Unknown
4,vla_shield.comp_05_legendary,Unknown


Ячейка 6: Привязка источников выпадения (Боссы)

In [6]:
with open(RAW_DATA_DIR / "NexusConfigStoreItemPoolList.json", "r", encoding="utf-8") as f:
    item_pool_list_data = json.load(f)

drop_sources = {}

def clean_handle(handle):
    if not handle or not isinstance(handle, str):
        return ""
    return handle.lower().replace("inv'", "").replace("'", "").strip()

# Сканируем всю базу пулов добычи боссов
for list_key, list_val in item_pool_list_data.items():
    boss_name = list_key.replace("ItemPoolList_", "").replace("_", " ").title()
    item_pools = list_val.get("fields", {}).get("ItemPools", [])
    
    for pool_entry in item_pools:
        itempool = pool_entry.get("itempool", {})
        item_data = itempool.get("item", {})
        
        if item_data.get("bInstance") and "Instance" in item_data:
            instance = item_data["Instance"] or {}
            items_in_pool = instance.get("items", [])
            for pool_item in items_in_pool:
                inner_item = pool_item.get("item", {}).get("item", {})
                handle = inner_item.get("Handle")
                if handle:
                    cleaned_h = clean_handle(handle)
                    if cleaned_h:
                        drop_sources.setdefault(cleaned_h, []).append(boss_name)
        else:
            handle = item_data.get("Handle")
            if handle:
                cleaned_h = clean_handle(handle)
                if cleaned_h:
                    drop_sources.setdefault(cleaned_h, []).append(boss_name)

def get_drop_source(row):
    cleaned_code = clean_handle(row["Item_Code"])
    if not cleaned_code:
        return "-"
    sources = drop_sources.get(cleaned_code, [])
    return ", ".join(set(sources)) if sources else "-"

df["Drop_Source"] = df.apply(get_drop_source, axis=1)
print("Источники добычи щитов успешно привязаны!")

Источники добычи щитов успешно привязаны!


Ячейка 7: Модульный регистронезависимый разбор слотов и классификация

In [7]:
part_brand_map = {
    "jak": "Jakobs", "ted": "Tediore", "hyp": "Hyperion", "cov": "CoV",
    "borg": "Ripper", "bor": "Ripper", "tor": "Torgue", "mal": "Maliwan",
    "vla": "Vladof", "atl": "Atlas"
}

def get_part_info(part_code, weapon_manufacturer, translation_map):
    code_lower = part_code.lower()
    cleaned_code = part_code.replace("part_", "")
    
    part_mfr = None
    for suffix, brand_name in part_brand_map.items():
        if f"_{suffix}" in code_lower or f"_{suffix}_" in code_lower:
            part_mfr = brand_name
            break
            
    if not part_mfr:
        part_mfr = weapon_manufacturer
        
    model_name = translation_map.get(part_code.lower(), translation_map.get(cleaned_code.lower(), None))
    
    if model_name:
        if any(brand in model_name for brand in ["Daedalus", "Atlas", "Hyperion", "Tediore", "Ripper", "Jakobs", "Order"]):
            return model_name
        return f"{part_mfr} ({model_name})"
    else:
        # Резервный перевод при отсутствии локализации
        display_code = cleaned_code
        for suffix in part_brand_map.keys():
            display_code = re.sub(rf"_{suffix}\b", "", display_code, flags=re.IGNORECASE)
            display_code = re.sub(rf"\b{suffix}_", "", display_code, flags=re.IGNORECASE)
        
        display_code = display_code.replace("_", " ").title()
        return f"{part_mfr} ({display_code})"

def extract_parts_by_slot_advanced(selection_rules, slot_key, category, weapon_manufacturer, translation_map, category_parts_by_slot):
    parts_list = []
    slot_key_lower = slot_key.lower()
    
    # Приводим оригинальные правила спавна к нижнему регистру для регистронезависимости
    rules_lower = {}
    if isinstance(selection_rules, dict):
        rules_lower = {k.lower(): v for k, v in selection_rules.items()}
    
    if slot_key_lower in rules_lower:
        slot_val = rules_lower[slot_key_lower]
        parts_in_slot = slot_val.get("parts", [])
        for p in parts_in_slot:
            part_code = p.get("part", "")
            if part_code:
                formatted_part = get_part_info(part_code, weapon_manufacturer, translation_map)
                parts_list.append(formatted_part)
    else:
        inherited_parts = category_parts_by_slot.get(category, {}).get(slot_key_lower, [])
        for part_code in inherited_parts:
            formatted_part = get_part_info(part_code, weapon_manufacturer, translation_map)
            parts_list.append(formatted_part)
            
    return ", ".join(sorted(list(set(parts_list)))) if parts_list else "-"

# --- ШАГ 1: Группируем все физические детали по оригинальным слотам щитов ---
category_parts_by_slot = {}
for category, cat_val in inventory_data.items():
    if not isinstance(cat_val, dict) or "shield" not in category.lower():
        continue
    parts_dict = cat_val.get("parts", {})
    if not isinstance(parts_dict, dict):
        continue
        
    category_parts_by_slot[category] = {}
    for part_id, part_val in parts_dict.items():
        if isinstance(part_val, dict):
            part_name = part_val.get("name", "")
            if isinstance(part_name, str) and part_name.startswith("part_"):
                slot_name = part_val.get("dependency_slot", "")
                if slot_name:
                    slot_name_lower = slot_name.lower()
                    category_parts_by_slot[category].setdefault(slot_name_lower, []).append(part_name)

# --- ШАГ 2: Реализуем наследование правил генерации щитов ---
def merge_shield_inheritance_rules(row, inventory_data):
    selection_rules = row.get("Selection_Rules", {})
    category = row.get("Internal_Category", "")
    all_rules = {}
    
    # Наследуем базовые правила обычных качеств
    category_parts = inventory_data.get(category, {}).get("parts", {})
    base_templates = ["comp_01_common", "comp_02_uncommon", "comp_03_rare", "comp_04_epic"]
    
    if isinstance(category_parts, dict):
        for part_key, part_val in category_parts.items():
            if isinstance(part_val, dict):
                part_name = part_val.get("name", "")
                if part_name in base_templates:
                    rules = part_val.get("fields", {}).get("PartTypeSelectionRules", {})
                    if isinstance(rules, dict):
                        all_rules.update(rules)
                        
    if isinstance(selection_rules, dict):
        all_rules.update(selection_rules)
    return all_rules

df["Merged_Rules"] = df.apply(lambda row: merge_shield_inheritance_rules(row, inventory_data), axis=1)

# --- ШАГ 3: Заполнение 10 категорий слотов щитов ---
all_game_slots = [
    "body", "body_acc", "element", "endgame", "firmware", 
    "pearl_elem", "pearl_stat", "primary_augment", "secondary_augment", "unique"
]

for slot in all_game_slots:
    df[slot] = df.apply(lambda row: extract_parts_by_slot_advanced(
        row["Merged_Rules"], 
        slot, 
        row["Internal_Category"],
        row["Manufacturer"], 
        part_translation_map,
        category_parts_by_slot
    ), axis=1)

# --- ШАГ 4: Парсинг стихий и определение типов щитов ---
def determine_shield_subtype(row):
    mfr = row["Manufacturer"]
    # Torgue, Vladof, Tediore специализируются на броне
    if mfr in ["Torgue", "Vladof", "Tediore"]:
        return "Armor Shield"
    # Остальные по умолчанию энергетические
    return "Energy Shield"

df["Shield Type"] = df.apply(determine_shield_subtype, axis=1)

def extract_shield_element(row):
    rules = row["Merged_Rules"]
    rules_lower = {k.lower(): v for k, v in rules.items()} if isinstance(rules, dict) else {}
    
    detected_elements = []
    if "element" in rules_lower:
        parts_list = rules_lower["element"].get("parts", [])
        for p in parts_list:
            p_code = p.get("part", "").lower()
            if "corrosive" in p_code: detected_elements.append("Corrosive")
            elif "cryo" in p_code: detected_elements.append("Cryo")
            elif "fire" in p_code: detected_elements.append("Fire")
            elif "shock" in p_code: detected_elements.append("Shock")
            elif "radiation" in p_code: detected_elements.append("Radiation")
            elif "normal" in p_code: detected_elements.append("Kinetic")
            
    return ", ".join(sorted(list(set(detected_elements)))) if detected_elements else "-"

df["Elements"] = df.apply(extract_shield_element, axis=1)
print("Все модули, стихии и типы щитов успешно обработаны!")

Все модули, стихии и типы щитов успешно обработаны!


Ячейка 8: Форматирование и выгрузка в CSV-таблицу

In [8]:
final_df = df.copy()

# Переименовываем базовые колонки
final_df = final_df.rename(columns={
    "Display_Name": "Name",
    "Drop_Source": "Drop Source",
    "World_Drop": "World Drop",
    "Item_Code": "Item Code"
})

# Умная фильтрация родительских пустых шаблонов
is_template = final_df["Item Code"].str.lower().str.endswith(("comp_05_legendary", "comp_06_pearl", "comp_06_pearlescent"))
final_df = final_df[~is_template]

# Порядок вывода столбцов
all_game_slots = [
    "body", "body_acc", "element", "endgame", "firmware", 
    "pearl_elem", "pearl_stat", "primary_augment", "secondary_augment", "unique"
]

columns_order = [
    "Item Code", "Name", "Rarity", "Manufacturer", "Shield Type", 
    "World Drop", "Drop Source", "Elements"
] + all_game_slots

final_df = final_df[columns_order]

# Сохраняем результат
final_df.to_csv(output_file, index=False, encoding="utf-8")

print(f"Экспорт щитов завершен! Таблица сохранена в: {output_file}")
print(f"Размерность: {final_df.shape[0]} строк на {final_df.shape[1]} колонок.")

Экспорт щитов завершен! Таблица сохранена в: ../data/output/shields_result_20260726_185131.csv
Размерность: 21 строк на 18 колонок.
